# Speech Coach — GPU Backend Server

**Run this notebook on Colab with a T4 GPU runtime.**

It starts a FastAPI server that your local webapp calls for pipeline processing.

### Setup (one-time):
1. Open this notebook in Google Colab
2. Runtime → Change runtime type → **T4 GPU**
3. **Run All** (Ctrl+F9)
4. Copy the ngrok URL printed at the bottom
5. Paste it in your webapp Settings → COLAB_BACKEND_URL

After that, all uploads from `localhost:3000` are processed on the T4 GPU automatically.

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers librosa parselmouth
!pip install -q openai-whisper
!pip install -q opencv-python mediapipe ultralytics
!pip install -q spacy textstat sentence-transformers
!pip install -q fastapi uvicorn python-multipart pyngrok
!python -m spacy download en_core_web_sm -q
print('✅ Dependencies installed')

In [ ]:
#@title 2. Clone / Update Repo
import os

REPO_URL = 'https://github.com/anvay-cpu/voice-analysis-pipeline.git'
REPO_DIR = '/content/voice-analysis-pipeline'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'✅ Repo ready at {REPO_DIR}')

In [ ]:
#@title 3. Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print('✅ T4 GPU ready')
else:
    print('⚠️ No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
#@title 4. Setup ngrok tunnel
#@markdown Get a free auth token at https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = '' #@param {type:"string"}

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print('✅ ngrok authenticated')
else:
    print('⚠️ No ngrok token — tunnel will work but may be rate-limited')
    print('   Get a free token at: https://dashboard.ngrok.com/signup')

In [ ]:
#@title 5. Pre-load Models (warm up GPU)
import sys
sys.path.insert(0, REPO_DIR)

print('Loading voice pipeline models...')
try:
    from src.pipeline import VoiceAnalysisPipeline
    _voice = VoiceAnalysisPipeline()
    print('  ✅ Voice pipeline ready')
except Exception as e:
    print(f'  ⚠️ Voice pipeline: {e}')

print('Loading body pipeline models...')
try:
    from src.body.pipeline import BodyAnalysisPipeline
    _body = BodyAnalysisPipeline()
    print('  ✅ Body pipeline ready')
except Exception as e:
    print(f'  ⚠️ Body pipeline: {e}')

print('Loading content pipeline...')
try:
    from src.content.pipeline import ContentAnalysisPipeline
    _content = ContentAnalysisPipeline()
    print('  ✅ Content pipeline ready')
except Exception as e:
    print(f'  ⚠️ Content pipeline: {e}')

print('\n✅ All models loaded on GPU')

In [ ]:
#@title 6. Start GPU Backend Server 🚀
#@markdown This cell runs forever — the server stays up as long as Colab is connected.

import threading
import uvicorn
from pyngrok import ngrok

# Import the API server
from src.api_server import app

# Create ngrok tunnel
public_url = ngrok.connect(8000, 'http')

print('=' * 60)
print('  🚀 SPEECH COACH GPU BACKEND IS LIVE')
print('=' * 60)
print(f'')
print(f'  📡 Public URL: {public_url}')
print(f'')
print(f'  Paste this URL in your webapp:')
print(f'  Settings → COLAB_BACKEND_URL → {public_url}')
print(f'')
print(f'  Or set in browser console:')
print(f'  localStorage.setItem("colab_url", "{public_url}")')
print(f'')
print('=' * 60)
print('  Server running... (keep this tab open)')
print('=' * 60)

# Run uvicorn (blocking)
uvicorn.run(app, host='0.0.0.0', port=8000)